# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [3]:
# TODO (hint: .str.lower().str.contains('pop'))
pop_mask = cat['product_name'].str.lower().str.contains('pop')

print(f'Pop-Tart count: {pop_mask.sum()}')
cat[pop_mask]

Pop-Tart count: 3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [4]:
# TODO
strawberry_mask = pop_mask & cat['product_name'].str.lower().str.contains('strawberry')

print('SKUs that are strawberry Pop-Tarts:')
cat.loc[strawberry_mask, ['sku', 'product_name']]

SKUs that are strawberry Pop-Tarts:


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry


**Pop-Tart rows that are a different product:** In this sample, there aren't any other Pop-Tart flavors because all three rows are for strawberry. However, the sample is very small, so it doesn't show the full range of products that might be in the catalog. In the Mini-ETL studio dataset, there was also a POPTART-BLUE SKU. The real midterm catalog probably includes other flavors, such as blueberry and S'mores. These should stay separate instead of being grouped under strawberry just because they're all Pop-Tarts. Being part of the same product line doesn't mean they're the same product.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [5]:
# TODO
CANON = {
    'PT-12': 'POPTART-STRAWBERRY',
    'SKU#459812': 'POPTART-STRAWBERRY',
    'POPTART-STRAW': 'POPTART-STRAWBERRY',
}

# .map() alone turns any SKU NOT in CANON into NaN -- .fillna() with the
# original column puts those unmapped SKUs back as themselves.
cat['canonical_sku'] = cat['sku'].map(CANON).fillna(cat['sku'])

cat[['sku', 'product_name', 'canonical_sku']]

,sku,product_name,canonical_sku
0,PT-12,pop-tarts strawberry,POPTART-STRAWBERRY
1,SKU#459812,Strawberry Pop-Tart,POPTART-STRAWBERRY
2,POPTART-STRAW,Pop-Tarts Strawberry,POPTART-STRAWBERRY


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [6]:
# TODO
before = sorted(cat['category'].unique())

cat['category'] = cat['category'].astype(str).str.strip().str.title()

after = sorted(cat['category'].unique())

print('before:', before)
print('after: ', after)
print()
print(cat['category'].value_counts())


before: ['Breakfast', 'Breakfast Foods']
after:  ['Breakfast', 'Breakfast Foods']

category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


Using .strip() and .title() didn't change anything because both category names were already formatted correctly. The issue is that "Breakfast" and "Breakfast Foods" might mean the same thing. Fixing this requires a separate mapping, like the SKU mapping in Q3. However, I'd confirm whether they should be combined before merging them.


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [7]:
# TODO
import re

def norm_key(name):
    """lowercase, strip punctuation, sort the words -- so word order and
    separators ('-' vs ' ') stop mattering for matching purposes."""
    cleaned = re.sub(r'[^a-z0-9\s]', ' ', name.lower())
    words = sorted(cleaned.split())
    return ' '.join(words)

cat['_name_key'] = cat['product_name'].apply(norm_key)
print(cat[['sku', 'product_name', '_name_key']])
print()

groups = cat.groupby('_name_key')['sku'].apply(list)
fragmented = groups[groups.apply(len) > 1]

print('candidate fragmented groups (same normalized name, different SKU):')
fragmented

             sku          product_name             _name_key
0          PT-12  pop-tarts strawberry  pop strawberry tarts
1     SKU#459812   Strawberry Pop-Tart   pop strawberry tart
2  POPTART-STRAW  Pop-Tarts Strawberry  pop strawberry tarts

candidate fragmented groups (same normalized name, different SKU):


,sku
_name_key,
pop strawberry tarts,"[PT-12, POPTART-STRAW]"


The automatic method only groups two of the three strawberry rows (PT-12 and POPTART-STRAW). SKU#459812 uses "Strawberry Pop-Tart" instead of "Strawberry Pop-Tarts," so its normalized key is slightly different and it ends up in a separate group.

This shows a limitation of word-sorting: it can miss differences in plurals, abbreviations, and product names. It's useful for finding possible matches, but the results still need to be checked manually, which is why Q2 asks for a hand-check.


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
# TODO
assert cat['canonical_sku'].notna().all(), 'mapping produced a null -- something got dropped instead of kept as itself'
assert len(cat) == len(cat.drop_duplicates(subset=['sku'])) or True  # row count check below is the real one
original_row_count = 3  # from the fallback sample read in cell 1; recompute if you're on the real file
assert len(cat) == original_row_count, f'expected {original_row_count} rows, got {len(cat)}'

print('mapping is safe: no nulls, no rows added or removed.')

mapping is safe: no nulls, no rows added or removed.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) I can't identify other product families in this three-row sample, but I would expect bottled water and branded clothing or merchandise to have similar issues in the full catalog. For example, I had already seen WATER-24 and BTL-WATER-24 used as separate SKUs for the same 24-pack of water. Clothing items could also have different SKUs depending on their size or how they were entered, such as "UVA T-Shirt" and "UVA Tee." I would handle these the same way as the Pop-Tarts: use the Q5 normalized-word-key method to find possible matches, then check each group manually as required in Q2. The automated method might miss differences in abbreviations or plural forms, just as it missed SKU#459812.

b) The difference between "Breakfast" and "Breakfast Foods" in Q4 is a good example of why not every category difference should be automatically fixed. From a formatting perspective, they're already cleaned up by .strip().title(), so the issue isn't really string cleaning. The question is whether the two categories should be combined. "Breakfast Foods" might include a wider range of products, like cereal and granola bars, while "Breakfast" could refer to a smaller group of grab-and-go items. Combining them could remove a distinction that was intentional. Since the data alone doesn't tell me whether these categories mean the same thing, I would ask the store's merchandising or inventory manager before merging them. That decision depends on how the business wants to organize its products, not just on what a cleaning script can change.